### Notebook to generate the optimal combination of cmems of OSCAR. 

- Gives best case if we are able to alway pick which model gives the best forecast. 
- Gives dataset in the format of all othe other forecasts
- 

In [2]:
import pandas as pd 
import numpy as np 

In [17]:
ds = pd.read_csv(r"saved_output\combined_cmems2024.csv") #combined_cmems2024.csv
ds1 = pd.read_csv(r"saved_output\OSCAR_2024v2.csv")

In [6]:
def haversine_df(df, lat1='lat1', lon1='lon1', lat2='lat2', lon2='lon2', radius=6371):
    """
    Compute the Haversine distance between two points for every row of a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing latitude/longitude columns
    lat1, lon1 : str
        Names of the first point's latitude/longitude columns
    lat2, lon2 : str
        Names of the second point's latitude/longitude columns
    radius : float
        Earth radius in meters (default is 6371000 m)

    Returns
    -------
    pandas.Series
        Distance (meters) for each row.
    """

    # Convert degrees → radians
    lat1_r = np.radians(df[lat1].values)
    lon1_r = np.radians(df[lon1].values)
    lat2_r = np.radians(df[lat2].values)
    lon2_r = np.radians(df[lon2].values)

    # Differences
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r

    # Haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return radius * c

In [21]:
ds['error_km'] = haversine_df(ds, "lat_true", "lon_true", "lat_forcast", "lon_forcast")
ds1['error_km'] = haversine_df(ds1, "lat_true", "lon_true", "lat_forcast", "lon_forcast")

def add_inital_time(ds:pd.DataFrame):
    """adds intial time of forecast (time - leadtime) and changes collumns to pd.DateTime objects"""
    ds["Time"] = pd.to_datetime(ds["Time"])
    ds["initial_time"] =  ds["Time"] - pd.to_timedelta(ds["leadtime"], unit = "hours")
    ds["initial_time"] = ds["initial_time"].dt.round(freq = "min")
    return ds
ds = add_inital_time(ds)
ds1 = add_inital_time(ds1)
bins = np.linspace(0,8*24,2*24+1)
ds1["lead_bins"] = pd.cut(ds1["leadtime"], bins)
ds ["lead_bins"] = pd.cut(ds["leadtime"], bins)
dsi = ds.query("leadtime == 0").reset_index(drop = True)
ds1i = ds1.query("leadtime == 0").reset_index(drop = True)
print(len(dsi), len(ds1i)) ## same amount of inital forecasts

8626 8626


In [24]:
##merge on BuoyID and inital_time add OSCAR Forcast times to cmems forecasts

# Remove duplicates, keeping first occurrence
ds_clean = ds.drop_duplicates(subset=['BuoyID', 'initial_time', 'leadtime'], keep='first')
ds1_clean = ds1.drop_duplicates(subset=['BuoyID', 'initial_time', 'leadtime'], keep='first')

merge = pd.merge(ds_clean, ds1_clean, 
                 left_on=['BuoyID', 'leadtime', 'initial_time'],
                 right_on=['BuoyID', 'leadtime', 'initial_time'], 
                 how='outer',
                 suffixes=('_cmems', '_OSCAR'))
# merge= pd.merge(ds, ds1, 
#                      left_on=['BuoyID', 'leadtime', 'initial_time'],
#                      right_on=['BuoyID', 'leadtime', 'initial_time'], 
#                      how='outer',
#                      suffixes=('_cmems', '_OSCAR'))

print(f"max merging error(lat {(merge.lat_true_cmems - merge.lat_true_OSCAR).max()}")
print(f"Amount of intial forecasts {merge.query("leadtime == 0").shape[0]}")


max merging error(lat 0.0
Amount of intial forecasts 8429


In [25]:
# Check individual dataset sizes
print("CMEMS initial forecasts:", ds.query("leadtime == 0").shape[0])
print("OSCAR initial forecasts:", ds1.query("leadtime == 0").shape[0])

# Check for duplicates in each dataset
cmems_dups = ds.query("leadtime == 0").groupby(['BuoyID', 'initial_time']).size()
oscar_dups = ds1.query("leadtime == 0").groupby(['BuoyID', 'initial_time']).size()

print("\nCMEMS duplicates per BuoyID+initial_time:")
print(cmems_dups[cmems_dups > 1].head(10))

print("\nOSCAR duplicates per BuoyID+initial_time:")
print(oscar_dups[oscar_dups > 1].head(10))

# Expected unique combinations
unique_combos = ds.query("leadtime == 0")[['BuoyID', 'initial_time']].drop_duplicates()
print(f"\nUnique BuoyID+initial_time combinations: {len(unique_combos)}")

CMEMS initial forecasts: 8626
OSCAR initial forecasts: 8626

CMEMS duplicates per BuoyID+initial_time:
BuoyID      initial_time       
ISD+496037  2024-12-01 02:55:00    2
ISD+496040  2024-12-01 03:42:00    2
M3i+109629  2024-04-01 00:09:00    2
M3i+111938  2024-05-01 01:30:00    2
M3i+115368  2024-02-01 00:22:00    2
            2024-03-01 00:22:00    2
M3i+117211  2024-04-01 01:25:00    2
M3i+117227  2024-08-01 00:47:00    2
M3i+124719  2024-04-01 01:59:00    2
M3i+124784  2024-03-01 03:04:00    2
dtype: int64

OSCAR duplicates per BuoyID+initial_time:
BuoyID      initial_time       
ISD+496037  2024-12-01 02:55:00    2
ISD+496040  2024-12-01 03:42:00    2
M3i+109629  2024-04-01 00:09:00    2
M3i+111938  2024-05-01 01:30:00    2
M3i+115368  2024-02-01 00:22:00    2
            2024-03-01 00:22:00    2
M3i+117211  2024-04-01 01:25:00    2
M3i+117227  2024-08-01 00:47:00    2
M3i+124719  2024-04-01 01:59:00    2
M3i+124784  2024-03-01 03:04:00    2
dtype: int64

Unique BuoyID+initial_t

In [26]:
binlist = merge.lead_bins_cmems.unique()
binlist = binlist.sort_values()
binlist[17]

Interval(68.0, 72.0, closed='right')

In [27]:
# Define binlist and target_time outside the function (once, not per group)
binlist = merge.lead_bins_cmems.unique()
binlist = binlist.sort_values()
target_time = binlist[17]  # hour 68-72

def better_forecast(group): 
    maxcmems = group.lead_bins_cmems.max() >= target_time
    maxoscar = group.lead_bins_OSCAR.max() >= target_time
    
    if maxcmems and maxoscar:  # both forecasts longer than target time
        row = group.query("lead_bins_cmems == @target_time").reset_index(drop=True)
        if len(row) > 0:
            if row.iloc[0]["error_km_cmems"] < row.iloc[0]["error_km_OSCAR"]: 
                best = "cmems"
            else: 
                best = "OSCAR"
        else:
            # Fallback: pick the one with lower error
            best = "cmems" if group["error_km_cmems"].min() < group["error_km_OSCAR"].min() else "OSCAR"
    elif maxcmems: 
        best = "cmems"
    elif maxoscar: 
        best = "OSCAR"
    else: 
        if group.lead_bins_cmems.max() < group.lead_bins_OSCAR.max():
            best = "cmems"
        else:
            best = "OSCAR"
    
    group["lat_forcast_opt"] = group[f"lat_forcast_{best}"]
    group["lon_forcast_opt"] = group[f"lon_forcast_{best}"]
    group["model"] = best
    return group

forecast = merge.groupby(["BuoyID", "initial_time"]).apply(better_forecast).reset_index()

In [28]:
forecast_simple = forecast.loc[:, ["BuoyID","Time_cmems", "leadtime", "initial_time", "lat_true_cmems", 
                                   "lon_true_cmems", "lat_forcast_opt", "lon_forcast_opt", "model"]]
forecast_simple = forecast_simple.rename(columns={"Time_cmems": "Time", "lat_true_cmems": "lat_true", 
                                                  "lon_true_cmems": "lon_true", "lat_forcast_opt": "lat_forcast", 
                                                  "lon_forcast_opt": "lon_forcast"})


In [29]:
forecast_simple.to_csv(r"saved_output\optimal_cmems_OSCAR_2024.csv")

In [30]:
print(forecast_simple.query("model == 'cmems'").shape)
print(forecast_simple.query("model == 'OSCAR'").shape)


(99266, 9)
(155703, 9)
